# Notebook 05 - Coverage QoE on real drive-test data 

**Dataset**: Raca, Quinlan, Zahran, Sreenan, *Beyond Throughput: a 4G LTE Dataset with Channel and Context Metrics*, ACM MMSys 2018. Zenodo record 1219679, CC BY 4.0.
135 traces from two Irish operators, one sample per second, five mobility patterns.

**What is tested**: the four-KPI coverage formula (eq. 5: RSRP, RSRQ, SINR, distance), evaluated in Section 4 of the paper.
The outcomes are downlink bitrate during active download and CQI. Neither enters the score.

**Question**: does the multi-KPI coverage score rank samples by delivered throughput better than RSRP alone or Fixed RSRP classes?

In [1]:
import os, sys, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
sys.path.insert(0, os.path.abspath('../src'))
from qoe_framework import normalize_kpi, linear_norm_neg
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11}); sns.set_style('whitegrid')
FIG_DIR, RES_DIR, SEED = '../results/figures/', '../results/', 42

## Step 1 - Load and clean

In [2]:
frames = []
for f in sorted(glob.glob('../data/raw/d3_lte_ucc/Dataset/*/*.csv')):
    d = pd.read_csv(f, na_values=['-'])
    d['trace'] = os.path.basename(f)
    d['mobility'] = os.path.basename(os.path.dirname(f))
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)
for c in ['RSRP','RSRQ','SNR','CQI','DL_bitrate','ServingCell_Distance']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')
print('All rows:', len(raw), '| traces:', raw.trace.nunique())

# Keep LTE samples taken during an active download, with all four KPIs present
# and inside the 3GPP reporting ranges (RSRP -140..-44 dBm, RSRQ -19.5..-3 dB).
d3 = raw[(raw.NetworkMode == 'LTE') & (raw.State == 'D')].dropna(
        subset=['RSRP','RSRQ','SNR','CQI','DL_bitrate','ServingCell_Distance']).copy()
d3 = d3[d3.RSRP.between(-140, -44) & d3.RSRQ.between(-19.5, -3)
        & d3.ServingCell_Distance.between(0, 20000)]
d3['DL_Mbps'] = d3.DL_bitrate / 1000.0
print('Rows kept:', len(d3), '| traces:', d3.trace.nunique())
print(d3.groupby('mobility').size().to_string())
print(d3[['RSRP','RSRQ','SNR','ServingCell_Distance','CQI','DL_Mbps']].describe().round(1).T[['min','50%','max']])

All rows: 174523 | traces: 134


Rows kept: 102383 | traces: 128
mobility
bus            6252
car           48562
pedestrian    30234
static        11994
train          5341
                        min    50%      max
RSRP                 -132.0  -98.0    -53.0
RSRQ                  -19.0  -13.0     -3.0
SNR                   -19.0    5.0     32.0
ServingCell_Distance   32.3  680.8  19108.5
CQI                     1.0    8.0     15.0
DL_Mbps                 0.0    9.9    173.0


## Step 2 - Split by trace (70/30) so that test samples come from unseen drives

In [3]:
rng = np.random.RandomState(SEED)
traces = np.array(sorted(d3.trace.unique())); rng.shuffle(traces)
n_tr = int(0.7 * len(traces))
train = d3[d3.trace.isin(traces[:n_tr])].copy()
test  = d3[d3.trace.isin(traces[n_tr:])].copy()
print(f'Train: {len(train):,} rows, {train.trace.nunique()} traces | Test: {len(test):,} rows, {test.trace.nunique()} traces')

Train: 75,355 rows, 89 traces | Test: 27,028 rows, 39 traces


## Step 3 - Weight procedure of Sect. 3.4 (screening, then eq. 8)

Measured on the training traces only. `r_i` are the recommendation priorities of the paper.

In [4]:
KPIS = ['RSRP','SINR','RSRQ','DIST']
COL  = {'RSRP':'RSRP','SINR':'SNR','RSRQ':'RSRQ','DIST':'ServingCell_Distance'}
r_i  = {'RSRP':0.375,'SINR':0.292,'RSRQ':0.250,'DIST':0.083}
rho_i = {k: abs(spearmanr(train[COL[k]], train.DL_Mbps)[0]) for k in KPIS}
TAU, GAMMA = 0.15, 0.5
kept = [k for k in KPIS if rho_i[k] >= TAU]
blend = {k: GAMMA*r_i[k] + (1-GAMMA)*rho_i[k] for k in kept}
w_fit = {k: blend[k]/sum(blend.values()) for k in kept}
w_pub = {'RSRP':0.35,'RSRQ':0.25,'SINR':0.30,'DIST':0.10}
tab_w = pd.DataFrame({'r_i': r_i, 'abs_rho_train': rho_i, 'w_published': w_pub, 'w_fitted': w_fit}).round(3)
print(tab_w.to_string()); print('Retained after screening (tau = 0.15):', kept)
tab_w.to_csv(RES_DIR + 'tab_d3_weights.csv')

        r_i  abs_rho_train  w_published  w_fitted
RSRP  0.375          0.392         0.35     0.315
SINR  0.292          0.574         0.30     0.356
RSRQ  0.250          0.548         0.25     0.328
DIST  0.083          0.021         0.10       NaN
Retained after screening (tau = 0.15): ['RSRP', 'SINR', 'RSRQ']


## Step 4 - Scores

In [5]:
def z_all(df):
    return pd.DataFrame({
        'RSRP': normalize_kpi(df.RSRP, 'RSRP_dBm'),
        'RSRQ': normalize_kpi(df.RSRQ, 'RSRQ_dB'),
        'SINR': normalize_kpi(df.SNR,  'SINR_dB'),
        'DIST': linear_norm_neg(df.ServingCell_Distance, L=5000, U=0),
    }, index=df.index)

def score(df, w):
    z = z_all(df); tot = sum(w.values())
    return 100 * sum(w[k]/tot * z[k] for k in w)

def add_scores(df):
    df['s_rsrp_only'] = 100 * normalize_kpi(df.RSRP, 'RSRP_dBm')
    df['s_itu']       = pd.cut(df.RSRP, [-np.inf,-110,-95,-80,np.inf], labels=False)
    df['s_pub4']      = score(df, w_pub)
    df['s_pub3']      = score(df, {k: w_pub[k] for k in ['RSRP','RSRQ','SINR']})
    df['s_fit']       = score(df, w_fit)
    return df
train, test = add_scores(train), add_scores(test)
METHODS = [('Fixed RSRP classes','s_itu'), ('RSRP only (logistic)','s_rsrp_only'),
           ('Composite, eq. (5)','s_pub4'),
           ('Composite without distance','s_pub3'),
           ('Composite, weights fitted on training traces','s_fit')]

## Step 5 - Rank agreement with throughput and CQI on the held-out traces

95% intervals from 1,000 bootstrap resamples of whole traces.

In [6]:
def boot_ci(df, col, target, B=1000, seed=SEED):
    rs = np.random.RandomState(seed)
    groups = {t: g[[col, target]].values for t, g in df.groupby('trace')}
    keys = list(groups); out = []
    for _ in range(B):
        samp = np.vstack([groups[k] for k in rs.choice(keys, len(keys))])
        out.append(spearmanr(samp[:,0], samp[:,1])[0])
    return np.percentile(out, [2.5, 97.5])

rows = []
for name, col in METHODS:
    r_dl  = spearmanr(test[col], test.DL_Mbps)[0]; lo, hi = boot_ci(test, col, 'DL_Mbps')
    r_cqi = spearmanr(test[col], test.CQI)[0];     lo2, hi2 = boot_ci(test, col, 'CQI')
    # trace-level: mean score vs mean throughput per held-out trace
    g = test.groupby('trace')[[col, 'DL_Mbps']].mean()
    rows.append({'method': name, 'rho_DL': round(r_dl,3), 'DL_lo': round(lo,3), 'DL_hi': round(hi,3),
                 'rho_CQI': round(r_cqi,3), 'CQI_lo': round(lo2,3), 'CQI_hi': round(hi2,3),
                 'rho_DL_trace_mean': round(spearmanr(g[col], g.DL_Mbps)[0],3)})
res = pd.DataFrame(rows); print(res.to_string(index=False))
res.to_csv(RES_DIR + 'tab_d3_coverage_results.csv', index=False)

                                      method  rho_DL  DL_lo  DL_hi  rho_CQI  CQI_lo  CQI_hi  rho_DL_trace_mean
                          Fixed RSRP classes   0.320  0.176  0.479    0.492   0.402   0.588              0.362
                        RSRP only (logistic)   0.332  0.171  0.505    0.536   0.437   0.628              0.314
                          Composite, eq. (5)   0.542  0.422  0.648    0.784   0.717   0.819              0.536
                  Composite without distance   0.541  0.421  0.648    0.787   0.719   0.823              0.516
Composite, weights fitted on training traces   0.554  0.434  0.658    0.795   0.729   0.828              0.545


## Step 6 - Paired bootstrap: is the multi-KPI score better than RSRP alone?

In [7]:
def boot_diff(df, a, b, target, B=1000, seed=SEED):
    rs = np.random.RandomState(seed)
    groups = {t: g[[a, b, target]].values for t, g in df.groupby('trace')}
    keys = list(groups); out = []
    for _ in range(B):
        s = np.vstack([groups[k] for k in rs.choice(keys, len(keys))])
        out.append(spearmanr(s[:,0], s[:,2])[0] - spearmanr(s[:,1], s[:,2])[0])
    out = np.array(out); return out.mean(), np.percentile(out, [2.5, 97.5]), (out <= 0).mean()
for a in ['s_pub4', 's_pub3', 's_fit']:
    for tgt in ['DL_Mbps', 'CQI']:
        m, ci, p = boot_diff(test, a, 's_rsrp_only', tgt)
        print(f'{a:8s} minus RSRP-only, target {tgt:8s}: diff = {m:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]  share<=0: {p:.3f}')

s_pub4   minus RSRP-only, target DL_Mbps : diff = +0.203  95% CI [+0.089, +0.316]  share<=0: 0.000


s_pub4   minus RSRP-only, target CQI     : diff = +0.245  95% CI [+0.164, +0.336]  share<=0: 0.000


s_pub3   minus RSRP-only, target DL_Mbps : diff = +0.202  95% CI [+0.096, +0.312]  share<=0: 0.000


s_pub3   minus RSRP-only, target CQI     : diff = +0.248  95% CI [+0.167, +0.339]  share<=0: 0.000


s_fit    minus RSRP-only, target DL_Mbps : diff = +0.215  95% CI [+0.096, +0.336]  share<=0: 0.000


s_fit    minus RSRP-only, target CQI     : diff = +0.256  95% CI [+0.172, +0.350]  share<=0: 0.000


## Step 7 - Band definition

The band statistics reported in the paper are computed on all samples in Step 11.

In [8]:
def band(s): return np.where(s >= 90, 'Excellent', np.where(s >= 70, 'Good', np.where(s >= 50, 'Degraded', 'Poor')))
order = ['Poor','Degraded','Good','Excellent']

## Step 8 - By mobility pattern

In [9]:
rows = []
for mob, g in test.groupby('mobility'):
    if len(g) < 500: continue
    rows.append({'mobility': mob, 'n': len(g),
                 'rho_RSRP_only': round(spearmanr(g.s_rsrp_only, g.DL_Mbps)[0],3),
                 'rho_composite': round(spearmanr(g.s_pub4, g.DL_Mbps)[0],3)})
print(pd.DataFrame(rows).to_string(index=False))

  mobility     n  rho_RSRP_only  rho_composite
       bus  1811          0.411          0.495
       car 15270          0.342          0.423
pedestrian  4521          0.257          0.398
    static  4764          0.662          0.788
     train   662          0.283          0.422


## Step 9 - Single-KPI baselines on the held-out traces

Is the composite better than its best single input?

In [10]:
rows = []
for name, col in [('RSRP alone','RSRP'), ('RSRQ alone','RSRQ'), ('SINR alone','SNR'), ('Composite, eq. (5)','s_pub4')]:
    rows.append({'input': name, 'rho_DL': round(spearmanr(test[col], test.DL_Mbps)[0],3),
                 'rho_CQI': round(spearmanr(test[col], test.CQI)[0],3)})
print(pd.DataFrame(rows).to_string(index=False))
for tgt in ['DL_Mbps','CQI']:
    m, ci, p = boot_diff(test, 's_pub4', 'SNR', tgt)
    print(f'Ours minus SINR alone, target {tgt}: diff = {m:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]')

             input  rho_DL  rho_CQI
        RSRP alone   0.332    0.536
        RSRQ alone   0.533    0.663
        SINR alone   0.591    0.818
Composite, eq. (5)   0.542    0.784


Ours minus SINR alone, target DL_Mbps: diff = -0.047  95% CI [-0.084, -0.018]


Ours minus SINR alone, target CQI: diff = -0.035  95% CI [-0.058, -0.019]


## Step 10 - Repeated trace splits (seeds 0-29)

Fixed scores, design-choice variants, and two learned references trained on the training traces of each split (inputs RSRP, RSRQ, SINR).

In [11]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from qoe_framework import linear_norm_pos
all_traces = np.array(sorted(d3.trace.unique()))
d3s = add_scores(d3.copy())

def variants(df):
    z = z_all(df)
    zl = pd.DataFrame({'RSRP': linear_norm_pos(df.RSRP, -110, -80), 'RSRQ': linear_norm_pos(df.RSRQ, -15, -8),
                       'SINR': linear_norm_pos(df.SNR, 0, 20), 'DIST': z['DIST']}, index=df.index)
    return {'Fixed RSRP classes': df.s_itu, 'RSRP alone': df.s_rsrp_only, 'RSRQ alone': df.RSRQ, 'SINR alone': df.SNR,
            'Composite, eq. (5)': df.s_pub4,
            'Equal weights': 100*(z.RSRP + z.RSRQ + z.SINR)/3,
            'Linear normalization': 100*(0.35*zl.RSRP + 0.25*zl.RSRQ + 0.30*zl.SINR + 0.10*zl.DIST),
            'SINR and RSRQ only': 100*(0.30*z.SINR + 0.25*z.RSRQ)/0.55}
XC = ['RSRP', 'RSRQ', 'SNR']
rows = []
for seed in range(30):
    r = np.random.RandomState(seed); tr = all_traces.copy(); r.shuffle(tr)
    k = int(0.7*len(tr)); trn = d3s[d3s.trace.isin(tr[:k])]; te = d3s[d3s.trace.isin(tr[k:])]
    row = {name: spearmanr(sc, te.DL_Mbps)[0] for name, sc in variants(te).items()}
    row['Linear regression (learned)'] = spearmanr(LinearRegression().fit(trn[XC], trn.DL_Mbps).predict(te[XC]), te.DL_Mbps)[0]
    row['Gradient boosting (learned)'] = spearmanr(HistGradientBoostingRegressor(random_state=0).fit(trn[XC], trn.DL_Mbps).predict(te[XC]), te.DL_Mbps)[0]
    rows.append(row)
rep = pd.DataFrame(rows)
summ = pd.DataFrame({'mean': rep.mean(), 'sd': rep.std(ddof=1)}).round(3)
C = 'Composite, eq. (5)'
summ['ahead_of_RSRP'] = [(rep[c] > rep['RSRP alone']).sum() for c in rep.columns]
summ['ahead_of_SINR'] = [(rep[c] > rep['SINR alone']).sum() for c in rep.columns]
summ['ahead_of_composite'] = [(rep[c] > rep[C]).sum() for c in rep.columns]
print(summ.to_string())
print('Composite (logistic) ahead of linear normalization in', int((rep[C] > rep['Linear normalization']).sum()), 'of 30')
print('Equal weights ahead of composite in', int((rep['Equal weights'] > rep[C]).sum()), 'of 30')
rep.round(4).to_csv(RES_DIR + 'tab_d3_repeated_splits.csv', index_label='seed')
summ.to_csv(RES_DIR + 'tab_d3_repeated_splits_summary.csv')

                              mean     sd  ahead_of_RSRP  ahead_of_SINR  ahead_of_composite
Fixed RSRP classes           0.380  0.078              3              0                   0
RSRP alone                   0.398  0.082              0              2                   0
RSRQ alone                   0.553  0.029             29              6                   5
SINR alone                   0.582  0.039             28              0                  15
Composite, eq. (5)           0.593  0.036             30             15                   0
Equal weights                0.605  0.035             30             17                  27
Linear normalization         0.580  0.035             30             15                   0
SINR and RSRQ only           0.604  0.033             30             28                  19
Linear regression (learned)  0.581  0.036             28             15                  16
Gradient boosting (learned)  0.638  0.033             30             30         

## Step 11 - Single-split extras, filters, band statistics on all samples, figures

In [12]:
sens = pd.Series({name: round(spearmanr(sc, test.DL_Mbps)[0], 3) for name, sc in variants(test).items()}, name='rho_DL_seed42')
print(sens.to_string()); sens.to_csv(RES_DIR + 'tab_d3_sensitivity.csv')
lo, hi = boot_ci(test, 'SNR', 'DL_Mbps'); print('SINR alone, seed 42: 95%% CI [%.3f, %.3f]' % (lo, hi))
print('rho(SINR, CQI), all samples: %.3f' % spearmanr(d3s.SNR, d3s.CQI)[0])

# what each cleaning filter removes (applied in this order)
b0 = raw[(raw.NetworkMode == 'LTE') & (raw.State == 'D')].dropna(subset=['RSRP','RSRQ','SNR','CQI','DL_bitrate','ServingCell_Distance'])
m1 = b0.RSRP.between(-140, -44); m2 = b0.RSRQ.between(-19.5, -3); m3 = b0.ServingCell_Distance.between(0, 20000)
print('LTE download samples with all values present:', len(b0))
print('outside RSRP range:', int((~m1).sum()), '| outside RSRQ range:', int((~m2).sum()), '| distance above 20 km:', int((~m3).sum()), '| removed in total:', int((~(m1 & m2 & m3)).sum()), '| kept:', int((m1 & m2 & m3).sum()))

d3s['band'] = band(d3s.s_pub4)
tb_all = d3s.groupby('band').agg(n=('DL_Mbps','size'), median_DL=('DL_Mbps','median')).reindex(order)
tb_all['share_pct'] = (100*tb_all.n/len(d3s)).round(1); tb_all['median_DL'] = tb_all.median_DL.round(1)
print(tb_all.to_string()); tb_all.to_csv(RES_DIR + 'tab_d3_bands_all.csv')
fast = d3s[d3s.DL_Mbps >= 20]
print('Samples at >= 20 Mbps labelled Poor (%):', round((fast.band == 'Poor').mean()*100, 1))

plt.rcParams.update({'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8})
fig, ax = plt.subplots(figsize=(3.6, 2.3))
sns.boxplot(data=d3s, x='band', y='DL_Mbps', order=order, showfliers=False, color='0.75', linewidth=0.8, ax=ax)
ax.set_xlabel('Coverage band'); ax.set_ylabel('Downlink bitrate (Mbps)')
plt.tight_layout(); plt.savefig(FIG_DIR + 'd3_throughput_by_band.pdf', bbox_inches='tight'); plt.close()
fig, ax = plt.subplots(figsize=(4.4, 2.3))
cols = ['RSRP alone', 'SINR alone', C, 'Gradient boosting (learned)']
sns.boxplot(data=rep[cols].rename(columns={C: 'Composite', 'Gradient boosting (learned)': 'Gradient boosting'}), color='0.75', linewidth=0.8, ax=ax)
ax.set_ylabel('Spearman correlation')
plt.tight_layout(); plt.savefig(FIG_DIR + 'd3_repeated_splits.pdf', bbox_inches='tight'); plt.close()
print('figures written')

Fixed RSRP classes      0.320
RSRP alone              0.332
RSRQ alone              0.533
SINR alone              0.591
Composite, eq. (5)      0.542
Equal weights           0.550
Linear normalization    0.533
SINR and RSRQ only      0.590


SINR alone, seed 42: 95% CI [0.481, 0.682]
rho(SINR, CQI), all samples: 0.762
LTE download samples with all values present: 107577
outside RSRP range: 55 | outside RSRQ range: 611 | distance above 20 km: 4592 | removed in total: 5194 | kept: 102383
               n  median_DL  share_pct
band                                  
Poor       69700        5.8       68.1
Degraded   20673       20.1       20.2
Good       11506       23.1       11.2
Excellent    504       29.1        0.5
Samples at >= 20 Mbps labelled Poor (%): 37.8


figures written


## Step 12 - Robustness: learned models on the single split, penalties, and the larger three-KPI sample

In [13]:
# learned references on the seed-42 split
print('seed 42, linear regression: %.3f' % spearmanr(LinearRegression().fit(train[XC], train.DL_Mbps).predict(test[XC]), test.DL_Mbps)[0])
print('seed 42, gradient boosting: %.3f' % spearmanr(HistGradientBoostingRegressor(random_state=0).fit(train[XC], train.DL_Mbps).predict(test[XC]), test.DL_Mbps)[0])

# (a) coverage penalties of Sect. 3.2 applied: max(0, base - P)
def with_pen(df):
    P = 15*(df.RSRP < -110) + 10*(df.RSRQ < -15) + 10*(df.SNR < 0)
    return np.maximum(0, df.s_pub4 - P)
d3s['s_pen'] = with_pen(d3s)
test['s_pen'] = with_pen(test); lo, hi = boot_ci(test, 's_pen', 'DL_Mbps')
print('seed 42, composite with penalties: %.3f, 95%% CI [%.3f, %.3f]' % (spearmanr(test.s_pen, test.DL_Mbps)[0], lo, hi))
print('penalty fires on (%%): RSRP %.1f, RSRQ %.1f, SINR %.1f' % (100*(d3s.RSRP < -110).mean(), 100*(d3s.RSRQ < -15).mean(), 100*(d3s.SNR < 0).mean()))
pen = []
for seed in range(30):
    r = np.random.RandomState(seed); tr = all_traces.copy(); r.shuffle(tr)
    te = d3s[d3s.trace.isin(tr[int(0.7*len(tr)):])]
    pen.append((spearmanr(te.s_pen, te.DL_Mbps)[0], spearmanr(te.s_pub4, te.DL_Mbps)[0], spearmanr(te.SNR, te.DL_Mbps)[0]))
pen = pd.DataFrame(pen, columns=['penalised', 'unpenalised', 'SINR'])
print('with penalties: mean %.3f (sd %.3f); ahead of unpenalised in %d, of SINR alone in %d of 30' % (pen.penalised.mean(), pen.penalised.std(ddof=1), (pen.penalised > pen.unpenalised).sum(), (pen.penalised > pen.SINR).sum()))

# (b) larger sample: no distance requirement
lte = raw[(raw.NetworkMode == 'LTE') & (raw.State == 'D')]
print('LTE download samples:', len(lte), '| missing distance:', int(lte.ServingCell_Distance.isna().sum()))
b = lte.dropna(subset=['RSRP', 'RSRQ', 'SNR', 'DL_bitrate']).copy()
b = b[b.RSRP.between(-140, -44) & b.RSRQ.between(-19.5, -3)]
b['DL_Mbps'] = b.DL_bitrate/1000.0
zb = pd.DataFrame({'RSRP': normalize_kpi(b.RSRP, 'RSRP_dBm'), 'RSRQ': normalize_kpi(b.RSRQ, 'RSRQ_dB'), 'SINR': normalize_kpi(b.SNR, 'SINR_dB')})
b['s3'] = 100*(0.35*zb.RSRP + 0.25*zb.RSRQ + 0.30*zb.SINR)/0.90
b['s_rsrp'] = 100*zb.RSRP
print('samples with the three radio KPIs:', len(b), '| traces:', b.trace.nunique())
tb = np.array(sorted(b.trace.unique())); big = []
for seed in range(30):
    r = np.random.RandomState(seed); tr = tb.copy(); r.shuffle(tr)
    te = b[b.trace.isin(tr[int(0.7*len(tr)):])]
    big.append((spearmanr(te.s3, te.DL_Mbps)[0], spearmanr(te.s_rsrp, te.DL_Mbps)[0], spearmanr(te.SNR, te.DL_Mbps)[0]))
big = pd.DataFrame(big, columns=['composite3', 'RSRP', 'SINR'])
print(big.agg(['mean', lambda s: s.std(ddof=1)]).round(3).to_string())
print('three-KPI composite ahead of RSRP in %d and of SINR in %d of 30' % ((big.composite3 > big.RSRP).sum(), (big.composite3 > big.SINR).sum()))
rb = pd.DataFrame({'penalised_mean': [pen.penalised.mean()], 'penalised_sd': [pen.penalised.std(ddof=1)], 'penalised_ahead_of_unpenalised': [(pen.penalised > pen.unpenalised).sum()], 'penalised_ahead_of_SINR': [(pen.penalised > pen.SINR).sum()],
                   'n_three_kpi': [len(b)], 'composite3_mean': [big.composite3.mean()], 'RSRP_mean': [big.RSRP.mean()], 'SINR_mean': [big.SINR.mean()],
                   'composite3_ahead_of_RSRP': [(big.composite3 > big.RSRP).sum()], 'composite3_ahead_of_SINR': [(big.composite3 > big.SINR).sum()]}).round(3)
rb.to_csv(RES_DIR + 'tab_d3_robustness.csv', index=False)
print('rounded means for the abstract:', rep.mean().round(4).to_dict())

seed 42, linear regression: 0.574


seed 42, gradient boosting: 0.648
penalty fires on (%): RSRP 13.0, RSRQ 12.0, SINR 25.4


with penalties: mean 0.612 (sd 0.038); ahead of unpenalised in 30, of SINR alone in 18 of 30
LTE download samples: 114326 | missing distance: 6745


samples with the three radio KPIs: 113528 | traces: 132


          composite3   RSRP   SINR
mean           0.588  0.406  0.552
<lambda>       0.039  0.059  0.038
three-KPI composite ahead of RSRP in 30 and of SINR in 23 of 30
rounded means for the abstract: {'Fixed RSRP classes': 0.3798, 'RSRP alone': 0.3976, 'RSRQ alone': 0.553, 'SINR alone': 0.582, 'Composite, eq. (5)': 0.5928, 'Equal weights': 0.6048, 'Linear normalization': 0.5799, 'SINR and RSRQ only': 0.6042, 'Linear regression (learned)': 0.5806, 'Gradient boosting (learned)': 0.6382}
